# Rollout cache explorer

Onboarding notebook for analysing the autoregressive rollout caches. Everything the
model generated during a rollout run lives in one HDF5 file per run; this notebook loads
one, shows what is inside, and gives you working starting points for plotting and peak
analysis. All the heavy lifting (reading the cache, re-deriving the real traces, building
the interactive browser and the horizon figures) is imported from the codebase, so this
file is mostly glue plus a clear space at the bottom for your own work.

## What a rollout is

The model generates one 256-sample future window from a real history window plus the real
control covariates. A *rollout* chains that: start at some fraction of a shot, generate a
window, feed the generated window back as the next history, and repeat to the end of the
shot. The control covariates and the physical time axis always come from the real shot;
only the observable history is generated. So a rollout is the model running free on its
own output, and the question is how fast and in what way it drifts from reality.

Each rollout is done for several *start fractions* (e.g. 5%, 10%, 25%, 50%, 75% of the
shot) and, in the newer runs, several stochastic *samples* per start point (the flow model
is stochastic, so the same start gives different rollouts).

## What is in the cache

One file per run, named `{run_name}_rollout.h5`, group layout `{shot}/{start_idx}/{sample_idx}`.
Each leaf holds three arrays:

- `generated_x` `(channels, T)`, float32, **normalized [0,1]**. The generated observables.
- `surr_labels_gen` `(history_length + T,)`, int16. Surrogate mode labels (FNOLSTM classifier)
  on the generated trace, over history + rollout.
- `surr_labels_real` same shape, the surrogate labels on the real trace over the same span.

Plus per-leaf attrs (`start_frac`, `start_i`, `t_start`, `t_end`, `n_windows`, `seq_length`,
`history_length`, `step`) and root attrs (`start_fractions`, `n_samples`, `cols_x`, `run_name`).

Two things are deliberately **not** stored, and are re-derived from the parquet when needed:
the real observables/controls, and the true (non-surrogate) labels. The cache only holds
what the model produced, everything else is a positional slice of the shot dataframe.

**Label convention.** The surrogate labels are the classifier argmax and are **unshifted**:
`0 = L, 1 = D, 2 = H`, never Unknown. This is NOT the `+1`-shifted `LHD_label` convention
used elsewhere in the code. Index colour/name lists directly by these values.

**Normalized space.** `generated_x` and the re-derived real observables are both in the
min-max normalized `[0,1]` space (min/max from the train split). Peak prominences in the
config are set in this same space. Call `data_module.denormalize(...)` for physical units.

## The API you will use

| Import | What it does |
|---|---|
| `RolloutHDFCache(name, mode="r")` | Open a cache. `.get_root_attrs()`, `.list_rollouts()`, `.get_rollout(shot, start, sample)`. |
| `load_results_from_cache(cache, shots=, max_samples=)` | Read rollouts into `RolloutResult` objects, filtered cheaply before any array read. |
| `build_rollout_records(results, dm, step, ...)` | Flat: one dict per rollout, with the real traces + timeline attached. For stats and single-rollout figures. |
| `build_rollout_groups(results, dm, step, ...)` | Grouped by (shot, start point): overlays the stochastic samples. For the interactive browser. |
| `rollout_browser_plotly(groups, x_names, c_names)` | The interactive dropdown browser figure. |
| `export_horizon_analysis(model_records, ...)` | Error-vs-depth figures and tables. |

Sibling notebooks do the production versions of each output: `rollout_browser.py`
(browser HTML), `paper_rollout.py` (per-rollout PDFs), `rollout_analysis.py` (horizon
figures/tables). This notebook is for exploring, they are for batch export.

## Setup: imports and repo root

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Put the repo root on sys.path so `import src` works no matter where the kernel started.
_REPO_ROOT = Path.cwd()
while not (_REPO_ROOT / "src").is_dir() and _REPO_ROOT != _REPO_ROOT.parent:
    _REPO_ROOT = _REPO_ROOT.parent
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))
os.chdir(_REPO_ROOT)  # so relative paths (output/, data/) resolve from the repo root
print("repo root:", _REPO_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.io as pio
from scipy.signal import find_peaks

from src.config import load_config_from_file
import src.data_loaders
from src.hdf_cache import RolloutHDFCache, get_cache_dir
from src.rollout import (
    load_results_from_cache,
    build_rollout_records,
    build_rollout_groups,
)
from src.plotters.rollout_plots import rollout_browser_plotly
from src.plotters.rollout_horizon import export_horizon_analysis
from src.metrics.metrics import batch_get_peakprops, get_peak_thresholds

pio.renderers.default = "notebook"  # inline plotly in the notebook

## Constants

Set these once. `CACHE_RUN` is the run name; the cache file is `{CACHE_RUN}_rollout.h5`
in `output/test_cache/` (or wherever `TEST_CACHE_DIR` points on the cluster). You can also
pass a full filename, with or without `.h5`, the resolver below handles it.

In [ ]:
# --- which cache -------------------------------------------------------------
# A run name / tag (its rollout cache is {name}_rollout.h5), or a full cache filename.
CACHE_RUN = "R-NormalMidAttSig03_anim"   # the paper CFM reeval; its rollout cache is the main target
# Offline alternative bundled in the repo (2 shots, 3 samples), if you have no cluster access:
# CACHE_RUN = "rollout_multisample_debug"

# --- what to look at ---------------------------------------------------------
SHOTS = None            # None = every shot in the cache, or a list like [57013, 61237, 64770]
MAX_SAMPLES = 10        # cap stochastic samples per start point (keep plots light)
MAX_SHOTS_IN_BROWSER = 10   # the interactive browser gets heavy past ~10 shots x 10 samples
START_FRACTION = None   # None = all start fractions, or a float like 0.50 to filter

# --- config knobs ------------------------------------------------------------
CONFIG_NAME = "plasmaflow"
OUTPUT_DIR = Path("output/coauthor")   # where this notebook writes html/pdf
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Snellius fetch (only used if the cache is missing locally AND you have access)
SNELLIUS_REMOTE = "snellius:/scratch-shared/mtresoor/final_cache"

## Selecting a cache, and fetching it from Snellius if you can

`resolve_cache_name` turns a run name / tag / filename into the cache stem (no `.h5`,
`_rollout` appended if needed). `ensure_cache` then makes sure the file is on disk: if it
is missing it will only try to rsync it from Snellius when an SSH connection actually works,
otherwise it prints the exact command so you can fetch it yourself or ask someone with
access. You most likely will not have Snellius access, so expect to be handed the `.h5`
files directly and drop them in `output/test_cache/`.

In [ ]:
def resolve_cache_name(x: str) -> str:
    """Run name / tag / filename -> cache stem used by RolloutHDFCache (no .h5)."""
    stem = x[:-3] if x.endswith(".h5") else x
    cache_dir = get_cache_dir()
    # Prefer an existing file; try the name as given, then with the _rollout suffix.
    for candidate in (stem, stem if stem.endswith("_rollout") else f"{stem}_rollout"):
        if (cache_dir / f"{candidate}.h5").exists():
            return candidate
    # Nothing on disk yet: default to the rollout-suffixed name for fetching.
    return stem if stem.endswith("_rollout") else f"{stem}_rollout"


def _snellius_reachable(timeout=6) -> bool:
    try:
        r = subprocess.run(
            ["ssh", "-o", "BatchMode=yes", "-o", f"ConnectTimeout={timeout}", "snellius", "true"],
            capture_output=True, timeout=timeout + 4,
        )
        return r.returncode == 0
    except Exception:
        return False


def ensure_cache(cache_stem: str) -> Path:
    """Return the local path to {cache_stem}.h5, fetching from Snellius only if possible."""
    path = get_cache_dir() / f"{cache_stem}.h5"
    if path.exists():
        print("cache present:", path)
        return path
    cmd = f"rsync -vz {SNELLIUS_REMOTE}/{cache_stem}.h5 {get_cache_dir()}/"
    if _snellius_reachable():
        print("fetching from Snellius:\n ", cmd)
        subprocess.run(cmd, shell=True, check=True)
    else:
        print(
            f"cache '{cache_stem}.h5' not found locally and Snellius is not reachable.\n"
            f"Get the file from someone with access (or run, if you have it):\n  {cmd}\n"
            f"then drop it in {get_cache_dir()}/"
        )
    return path


CACHE_NAME = resolve_cache_name(CACHE_RUN)
CACHE_PATH = ensure_cache(CACHE_NAME)
print("using cache:", CACHE_NAME)

## Config and data module

The data module is needed to re-derive the real observables/controls and the physical time
axis that the cache leaves out. `prepare_data` + `setup` load the parquet and compute the
train-split normalization. This is the slow step (a few seconds), run it once.

In [ ]:
C = load_config_from_file(CONFIG_NAME, as_omega=True)
DataModuleClass = getattr(src.data_loaders, C.data.Class)
data_module = DataModuleClass(**C.data)
data_module.prepare_data()
data_module.setup()

CHANNEL_NAMES = list(C.data.cols.x)          # observable channels, e.g. FIR, PD, DML, POHM, Z_axis
C_NAMES = list(C.data.cols.get("c", []))     # control covariates
PD_INDEX = CHANNEL_NAMES.index("PD")         # H-alpha photodiode, the ELM-bearing channel
PROMINENCE, ELM_PD_PROMINENCE = get_peak_thresholds(C)  # in normalized [0,1] space
print("channels:", CHANNEL_NAMES)
print("controls:", C_NAMES)
print(f"peak prominence (all channels)={PROMINENCE}, ELM-scale PD prominence={ELM_PD_PROMINENCE}")

## Overview of the cache

`list_rollouts()` returns every `(shot, start_idx, sample_idx)` triple without reading any
array, so it is cheap even for a big cache. Below: the root attrs, then a small table of
what is available per (shot, start point).

In [ ]:
cache = RolloutHDFCache(CACHE_NAME, mode="r")
root = cache.get_root_attrs()
print("run_name:      ", root.get("run_name"))
print("start_fractions:", list(root.get("start_fractions", [])))
print("n_samples:     ", root.get("n_samples"))
print("cols_x:        ", list(root.get("cols_x", [])))

keys = cache.list_rollouts()  # sorted (shot, start_idx, sample_idx)
print(f"\n{len(keys)} rollouts across {len({k[0] for k in keys})} shots")

# One row per (shot, start point): how many samples, and the rollout attrs.
rows = []
seen = set()
for shot, start_idx, sample_idx in keys:
    if (shot, start_idx) in seen:
        continue
    seen.add((shot, start_idx))
    a = cache.get_rollout(shot, start_idx, 0)  # attrs come along with the arrays
    n_samp = sum(1 for k in keys if k[0] == shot and k[1] == start_idx)
    rows.append({
        "shot": shot, "start_idx": start_idx, "n_samples": n_samp,
        "start_frac": round(float(a["start_frac"]), 3), "n_windows": int(a["n_windows"]),
        "t_start": round(float(a["t_start"]), 3), "t_end": round(float(a["t_end"]), 3),
        "gen_len_T": a["generated_x"].shape[-1],
    })
overview = pd.DataFrame(rows).sort_values(["shot", "start_frac"]).reset_index(drop=True)
overview

## Inspecting one rollout

`get_rollout` returns the arrays plus the attrs as a plain dict. Note the label arrays are
`history_length + T` long (they cover the real history window too), while `generated_x` is
only the generated part `T`.

In [ ]:
# Pick the first rollout in the cache (change these to target a specific one).
SHOT0, START0, SAMPLE0 = keys[0]
r = cache.get_rollout(SHOT0, START0, SAMPLE0)
print(f"shot {SHOT0}, start_idx {START0}, sample {SAMPLE0}")
print("generated_x:    ", r["generated_x"].shape, r["generated_x"].dtype, "(normalized [0,1])")
print("surr_labels_gen:", r["surr_labels_gen"].shape, "unique:", np.unique(r["surr_labels_gen"]))
print("surr_labels_real:", r["surr_labels_real"].shape, "unique:", np.unique(r["surr_labels_real"]))
print("attrs:", {k: v for k, v in r.items() if k not in ("generated_x", "surr_labels_gen", "surr_labels_real")})

## Attaching the real context: records

`load_results_from_cache` reads the rollouts (filtered by `shots`/`max_samples` before any
array is touched), and `build_rollout_records` attaches, per rollout, the real observables
(`real_x`), the controls (`real_c`) and the physical timeline (`times`) as positional slices
of the shot dataframe. The timeline spans history + rollout; `generated_x` aligns with
`times[history_length:]`.

`build_rollout_records` gives you a flat list (one dict per rollout), which is what the
single-rollout figures and the peak analysis below use. `build_rollout_groups` instead
groups the samples per start point, which is what the interactive browser wants.

In [ ]:
results = load_results_from_cache(cache, shots=SHOTS, max_samples=MAX_SAMPLES)
step = int(cache.get_rollout(*keys[0])["step"])
records = build_rollout_records(results, data_module, step=step, shots=SHOTS)
if START_FRACTION is not None:
    records = [rec for rec in records if abs(rec["start_frac"] - START_FRACTION) < 1e-6]
print(f"{len(records)} rollout records (step={step})")
rec = records[0]
print("record keys:", sorted(rec.keys()))
print("real_x:", rec["real_x"].shape, "| generated_x:", rec["generated_x"].shape,
      "| times:", rec["times"].shape, f"[{rec['times'][0]:.3f}..{rec['times'][-1]:.3f}]s")

## Plotting one rollout inline

A compact matplotlib view of a single rollout: every observable channel with the real
history (grey), the real future (black) and the generated trace (orange), the controls,
and the two surrogate mode-label bars (generated vs real). The vertical line is the rollout
start; dotted lines are the chained window boundaries.

For the polished paper PDFs use `eval_notebooks/paper_rollout.py`; this is the quick look.

In [ ]:
MODE_COLORS = ["lightskyblue", "orange", "red"]  # unshifted 0=L, 1=D, 2=H
MODE_NAMES = ["L", "D", "H"]


def plot_rollout_inline(record, figsize=(11, 9)):
    times = record["times"]
    hl = int(record["history_length"])
    t_start = float(record["t_start"])
    gen_times = times[hl:]
    step = int(record["step"])
    boundaries = [times[hl + b] for b in range(step, record["generated_x"].shape[-1], step)]
    n_x = len(CHANNEL_NAMES)

    fig, axes = plt.subplots(n_x + 3, 1, figsize=figsize, sharex=True,
                             gridspec_kw={"height_ratios": [1] * n_x + [0.6, 0.25, 0.25]})
    for ch in range(n_x):
        ax = axes[ch]
        ax.plot(times[:hl], record["real_x"][ch, :hl], color="0.55", lw=0.9, label="real history")
        ax.plot(gen_times, record["real_x"][ch, hl:], color="black", lw=0.8, label="real")
        ax.plot(gen_times, record["generated_x"][ch], color="#D55E00", lw=0.8, label="generated")
        for b in boundaries:
            ax.axvline(b, color="0.7", lw=0.4, ls=":")
        ax.axvline(t_start, color="0.25", lw=0.8)
        ax.set_ylabel(CHANNEL_NAMES[ch], rotation=0, ha="right", va="center")
        if ch == 0:
            ax.legend(loc="upper right", fontsize=7, ncol=3)

    ax_c = axes[n_x]
    for ci, name in enumerate(C_NAMES):
        ax_c.plot(times, record["real_c"][ci], lw=0.9, label=name)
    ax_c.axvline(t_start, color="0.25", lw=0.8)
    ax_c.set_ylabel("C", rotation=0, ha="right", va="center")
    ax_c.legend(loc="upper right", fontsize=6, ncol=len(C_NAMES))

    for ax, labels, name in ((axes[n_x + 1], record["surr_labels_gen"], "gen"),
                             (axes[n_x + 2], record["surr_labels_real"], "real")):
        # labels align 1:1 with times; colour each contiguous mode run
        vals = np.asarray(labels)
        change = np.flatnonzero(np.diff(vals)) + 1
        bounds = [0, *change, len(vals)]
        for s, e in zip(bounds[:-1], bounds[1:]):
            ax.axvspan(times[s], times[min(e, len(times) - 1)], color=MODE_COLORS[int(vals[s])], lw=0)
        ax.axvline(t_start, color="0.25", lw=0.8)
        ax.set_yticks([])
        ax.set_ylabel(name, rotation=0, ha="right", va="center")
    axes[-1].set_xlabel("Shot time (s)")
    fig.suptitle(f"Shot {record['shot_number']}  start {record['start_frac']:.0%}  "
                 f"t0={t_start:.2f}s  {record['n_windows']} windows  sample {record['sample_idx']}")
    fig.align_ylabels(axes)
    plt.show()
    return fig


_ = plot_rollout_inline(records[0])

## The interactive browser: many rollouts, one figure

`rollout_browser_plotly` builds one plotly figure with a dropdown, one entry per (shot,
start point), overlaying the stochastic samples. Keep it to about 10 shots and 10 samples;
past that the HTML gets heavy and slow to open. The `START_FRACTION` / `SHOTS` / `MAX_SAMPLES`
constants at the top control the subset. Written to an HTML file you can open in a browser,
and also shown inline.

In [ ]:
browser_shots = SHOTS
if browser_shots is None:
    all_shots = sorted({k[0] for k in keys})
    browser_shots = all_shots[:MAX_SHOTS_IN_BROWSER]  # cap for a responsive figure
elif len(browser_shots) > MAX_SHOTS_IN_BROWSER:
    browser_shots = list(browser_shots)[:MAX_SHOTS_IN_BROWSER]

browser_results = load_results_from_cache(cache, shots=browser_shots, max_samples=MAX_SAMPLES)
groups = build_rollout_groups(browser_results, data_module, step=step,
                              shots=browser_shots, max_samples=MAX_SAMPLES)
if START_FRACTION is not None:
    groups = [g for g in groups if abs(g["start_frac"] - START_FRACTION) < 1e-6]
print(f"{len(groups)} starting points in the browser "
      f"({len(browser_shots)} shots, <= {MAX_SAMPLES} samples each)")

fig = rollout_browser_plotly(groups, CHANNEL_NAMES, C_NAMES)
out_html = OUTPUT_DIR / f"rollouts_{CACHE_NAME}.html"
pio.write_html(fig, out_html)
print("wrote", out_html)
fig  # inline

## One shot and start fraction to PDF / inline image

For a single starting point (optionally a single sample), export the matplotlib figure to
PDF and show a raster preview inline. This is the per-rollout artifact; the batch version
over a whole cache is `eval_notebooks/paper_rollout.py`.

In [ ]:
from IPython.display import Image, display

TARGET_SHOT = overview.iloc[0]["shot"]        # change to any shot in the overview table
TARGET_FRAC = overview.iloc[0]["start_frac"]  # change to any start fraction for that shot
TARGET_SAMPLE = 0

match = [
    rec for rec in build_rollout_records(
        load_results_from_cache(cache, shots=[int(TARGET_SHOT)]), data_module, step=step)
    if abs(rec["start_frac"] - TARGET_FRAC) < 1e-6 and rec["sample_idx"] == TARGET_SAMPLE
]
if not match:
    print("no rollout for", TARGET_SHOT, TARGET_FRAC, "sample", TARGET_SAMPLE)
else:
    fig = plot_rollout_inline(match[0])
    pdf_path = OUTPUT_DIR / f"rollout_{TARGET_SHOT}_{TARGET_FRAC:.2f}_s{TARGET_SAMPLE}.pdf"
    jpg_path = pdf_path.with_suffix(".jpg")
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(jpg_path, dpi=110, bbox_inches="tight")
    plt.close(fig)
    print("wrote", pdf_path, "and", jpg_path)
    display(Image(filename=str(jpg_path)))

## Peak analysis: ELM timestamps and frequency

The point of the rollouts is whether the model reproduces the timing statistics of ELMs
(stochastic H-alpha bursts on the PD channel), not their exact positions. `peak_times`
returns the timestamps (in seconds) of peaks above a prominence on any channel, for both the
generated and the real trace over the rollout span. Prominence is in the normalized `[0,1]`
space, so `ELM_PD_PROMINENCE` (the config's ELM-scale threshold on PD) is the natural choice
for ELM peaks. From the timestamps you get counts, inter-peak intervals, and the ELM rate to
compare against ground truth.

For the full peak *properties* (height, width, prominence, base, energy) the codebase's
`batch_get_peakprops` gives the same `PeakProps` objects the thesis metrics use; a hook for
that is at the very bottom (the "full" peak-marker overlay).

In [ ]:
def peak_times(record, channel=PD_INDEX, prominence=ELM_PD_PROMINENCE, which="both"):
    """Timestamps (s) of peaks above `prominence` on one channel, over the rollout span.

    Prominence is in normalized [0,1] units (same space as generated_x). Returns
    (gen_times, real_times) arrays of seconds, or one of them if which is 'gen'/'real'.
    Peaks are found on the generated part only (times[history_length:]).
    """
    hl = int(record["history_length"])
    gen_times = record["times"][hl:]
    out = {}
    if which in ("gen", "both"):
        idx, _ = find_peaks(record["generated_x"][channel], prominence=prominence)
        out["gen"] = gen_times[idx]
    if which in ("real", "both"):
        idx, _ = find_peaks(record["real_x"][channel, hl:], prominence=prominence)
        out["real"] = gen_times[idx]
    return (out["gen"], out["real"]) if which == "both" else out[which]


# Example: pool ELM inter-peak intervals over the loaded records, generated vs real.
gen_intervals, real_intervals = [], []
gen_counts, real_counts = [], []
for rec in records:
    gt, rt = peak_times(rec, channel=PD_INDEX, prominence=ELM_PD_PROMINENCE)
    gen_counts.append(len(gt))
    real_counts.append(len(rt))
    gen_intervals.extend(np.diff(gt))
    real_intervals.extend(np.diff(rt))

print(f"PD ELM peaks/rollout: generated median {np.median(gen_counts):.0f}, "
      f"real median {np.median(real_counts):.0f}")

fig, (axc, axi) = plt.subplots(1, 2, figsize=(11, 3.4))
bins = np.arange(0, max(gen_counts + real_counts) + 2) - 0.5
axc.hist(real_counts, bins=bins, alpha=0.6, label="real", color="black")
axc.hist(gen_counts, bins=bins, alpha=0.6, label="generated", color="#D55E00")
axc.set_xlabel("ELM peaks per rollout"); axc.set_ylabel("rollouts"); axc.legend()
if real_intervals and gen_intervals:
    ibins = np.linspace(0, np.percentile(real_intervals + gen_intervals, 95), 40)
    axi.hist(real_intervals, bins=ibins, alpha=0.6, label="real", color="black", density=True)
    axi.hist(gen_intervals, bins=ibins, alpha=0.6, label="generated", color="#D55E00", density=True)
    axi.set_xlabel("inter-ELM interval (s)"); axi.set_ylabel("density"); axi.legend()
fig.suptitle("PD ELM timing: generated vs real")
plt.show()

## Plotting a rollout with peak markers

Two levels. **Simple** just drops a dot at every detected peak on the PD trace, generated
and real. **Full** reuses the thesis peak machinery (`batch_get_peakprops` + `add_peak_markers`)
to draw the height/width/base/energy markers on top of the signal. The full version is dense
and slower, and its x axis is the sample index (not shot time); use it for one channel of one
rollout when you want to inspect the peak properties themselves, not as a browsing view.

In [ ]:
def plot_peak_markers_simple(record, channel=PD_INDEX, prominence=ELM_PD_PROMINENCE):
    """Signal with a dot at each detected peak, generated (orange) vs real (black)."""
    hl = int(record["history_length"])
    gen_times = record["times"][hl:]
    gen = record["generated_x"][channel]
    real = record["real_x"][channel, hl:]
    gi, _ = find_peaks(gen, prominence=prominence)
    ri, _ = find_peaks(real, prominence=prominence)
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(gen_times, real, color="black", lw=0.7, label="real")
    ax.plot(gen_times, gen, color="#D55E00", lw=0.7, alpha=0.9, label="generated")
    ax.plot(gen_times[ri], real[ri], "o", color="black", ms=4)
    ax.plot(gen_times[gi], gen[gi], "o", color="#D55E00", ms=4)
    ax.set_title(f"{CHANNEL_NAMES[channel]} peaks (prominence {prominence}) "
                 f"- shot {record['shot_number']} @ {record['start_frac']:.0%}: "
                 f"gen {len(gi)}, real {len(ri)}")
    ax.set_xlabel("Shot time (s)"); ax.legend(loc="upper right", fontsize=8)
    plt.show()


plot_peak_markers_simple(records[0])

In [ ]:
import warnings
import plotly.graph_objects as go
from src.plotters.flow_plots import add_peak_markers


def plot_peak_markers_full(record, channel=PD_INDEX, prominence=None):
    """Dense peak-property markers via the thesis machinery, one channel, generated trace.

    Slow and busy: x is the sample index, and every peak gets height/width/base/energy
    markers. For close inspection of one rollout's peaks, not for browsing.
    """
    warnings.warn("plot_peak_markers_full is dense and slow; use it on one rollout/channel.")
    prominence = ELM_PD_PROMINENCE if prominence is None else prominence
    gen = record["generated_x"]  # (channels, T), normalized
    dml_index = CHANNEL_NAMES.index("DML") if "DML" in CHANNEL_NAMES else None
    peaks_per_channel = batch_get_peakprops(
        gen[None], prominence=PROMINENCE, dml_channel_index=dml_index,
        pd_channel_index=PD_INDEX, elm_pd_prominence=prominence,
    )[0]
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=gen[channel], mode="lines",
                             line=dict(color="#D55E00", width=1), name=CHANNEL_NAMES[channel]))
    add_peak_markers(
        fig, peaks_per_channel[channel], group="generated", shot_number=record["shot_number"],
        hover_info_template="x=%{x}, y=%{y:.3f}", channel_color="#0072B2",
        channel_name=CHANNEL_NAMES[channel],
    )
    fig.update_layout(height=400, title=f"{CHANNEL_NAMES[channel]} peak properties "
                      f"(shot {record['shot_number']} @ {record['start_frac']:.0%}, sample index axis)",
                      xaxis_title="sample index")
    return fig


# Uncomment to render the dense view for one rollout:
# plot_peak_markers_full(records[0])

## Optional: error vs rollout depth (horizon figures)

`export_horizon_analysis` produces the error-vs-depth figures and the csv/tex tables (median
+ IQR per start fraction and depth `k`, one line per model). This is the same code the run
produces automatically; here it is against the loaded cache. Pass several `(name, records)`
pairs to overlay models.

In [ ]:
horizon_records = build_rollout_records(
    load_results_from_cache(cache, shots=SHOTS), data_module, step=step, shots=SHOTS)
df = export_horizon_analysis(
    [(root.get("run_name") or CACHE_NAME, horizon_records)],
    channel_names=CHANNEL_NAMES,
    pd_index=PD_INDEX,
    elm_prominence=float(ELM_PD_PROMINENCE),
    pdf_dir=OUTPUT_DIR / "horizon",
    table_dir=OUTPUT_DIR / "horizon",
)
print("horizon dataframe:", df.shape)
df.head()

## How to hook in, and what you can analyse

Short version of the moving parts:

- The cache is read-only and self-contained: `RolloutHDFCache(name, "r")` plus
  `list_rollouts()` / `get_rollout(...)`. Nothing writes to it here.
- Everything real (observables, controls, time, true labels) is a positional slice of the
  shot dataframe, keyed by `start_idx`. `build_rollout_records` does that join for you; if you
  need something it does not attach, `data_module.data[data_module.data['ShotNum'] == shot]`
  is the raw per-shot dataframe (index = physical time in seconds), and
  `data_module.denormalize(x)` converts normalized arrays to physical units.
- `generated_x` aligns with `times[history_length:]`; the label arrays cover the full
  `times`. Labels are unshifted `0=L, 1=D, 2=H`.
- Prominence thresholds live in `C.evaluation.peaks` (via `get_peak_thresholds`) and are in
  normalized space.

Directions that are set up and ready:

- ELM timing: `peak_times` gives per-rollout peak timestamps; pool intervals/counts by start
  fraction, by shot, or by rollout depth and compare the generated and real distributions.
- Horizon behaviour: `export_horizon_analysis`, or its per-(rollout, k) dataframe, for how any
  metric degrades with autoregressive depth.
- Mode dynamics: the surrogate label arrays give L/D/H sequences for generated and real;
  transition counts and dwell times are a run-length encoding away.
- Full peak properties: `batch_get_peakprops` for height/width/prominence/base/energy per peak.

## Your analysis space

Everything above is loaded: `cache`, `records`, `groups`, `data_module`, `C`, and the helpers
`peak_times`, `plot_rollout_inline`, `plot_peak_markers_simple`. Start here.

In [ ]:
# scratch space for your own analysis